Week 4 (19/03/2025) - Billboard exercise

In [ ]:
import numpy as np
import cv2

# Start by defining the function for clicking on the billboard image at runtime.
def onClick(event, x, y, flags, params):
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(dst_points) < 4:
            dst_points.append([x, y])
            cv2.circle(img_copy, (x, y), 15, (0, 0, 255), -1)
            cv2.imshow("Click", img_copy)

# Load the images for the task.
base_img = cv2.imread("/home/gianmaria/01-Data/billboard.jpg") # billboard image
img_copy = base_img.copy()
img2 = cv2.imread("/home/gianmaria/01-Data/ezio.jpg") # image to place on the billboard
cv2.namedWindow("Img", cv2.WINDOW_KEEPRATIO)

# Get the image data to understand the size of the output image.
base_h, base_w = base_img.shape[:2]
img2_h, img2_w = img2.shape[:2]

# The problem now lies in getting the source points and the destination points for the transformation.
src_points = np.float32([[0, 0], [0, img2_h], [img2_w, img2_h], [img2_w, 0]]) # corners of the image to place on the billboard
dst_points = [] # destination points to obtain at runtime by clicking on the billboard

# Define the window to click on the billboard.
cv2.namedWindow("Click", cv2.WINDOW_KEEPRATIO)
cv2.setMouseCallback("Click", onClick)

# Show the image to be clicked on.
cv2.imshow("Click", base_img)
cv2.waitKey(0)
cv2.closeAllWindows()

# Compute the homography matrix in order to apply the transformation.
dst_float = np.float32(dst_points) # remember to convert the standard list into a numpy array
H = cv2.getPerspectiveTransform(src_points, dst_float)

# Apply the transformation to the image that must be placed over the billboard.
warped = cv2.warpPerspective(img2, H, (base_w, base_h)) # in the output image, all pixels around the billboard will be black due to the size differences

# Use the warped image to create a binary mask.
mask = np.zeros(base_img.shape, dtype=np.uint8)
cv2.fillConvexPoly(mask, np.int32(dst_float), (255, 255, 255)) # create a white rectangle in correspondence of the billboard